# SIH 2026 (SIH26103) - REAL PAIMANA MoSPI ML Training Pipeline (2001 - 2026)
### Predictive Infrastructure Monitoring & Early Warning Platform (MoSPI / DIID)

**100% REAL GOVERNMENT DATASET:** Trained on **16,088 real project monitoring records** extracted directly from official Ministry of Statistics & Programme Implementation (MoSPI) Flash Reports (2001–2026 series).

**Dataset File:** `paimana_real_2001_2026_dataset.csv`

**Data Split Strategy:**
- **80% Training Set**: Used to fit RandomForest & XGBoost regressors and classifiers
- **10% Test Evaluation Set**: Used to evaluate prediction metrics ($R^2$, MAE, RMSE)
- **10% Held-Out User Test Set**: Exported as `user_test_holdout.csv` for independent user verification!

In [ ]:
# Step 1: Install required libraries
!pip install -q pandas numpy scikit-learn xgboost joblib matplotlib seaborn

In [ ]:
# Step 2: Upload Real MoSPI PAIMANA Dataset (paimana_real_2001_2026_dataset.csv)
import os
import pandas as pd
from google.colab import files

dataset_path = "paimana_real_2001_2026_dataset.csv"
if not os.path.exists(dataset_path):
    print("[*] Please upload your official 'paimana_real_2001_2026_dataset.csv' file below:")
    uploaded = files.upload()
    for filename, content in uploaded.items():
        with open(filename, "wb") as f:
            f.write(content)
        print(f"[+] Uploaded real MoSPI dataset file: {filename}")

df_raw = pd.read_csv(dataset_path)
print(f"\n[+] Loaded REAL MoSPI PAIMANA Dataset: {df_raw.shape[0]} project rows, {df_raw.shape[1]} columns.")
df_raw.head()

In [ ]:
# Step 3: Zero-Leakage Feature Engineering on Real Government Data
import numpy as np

def engineer_real_features(df):
    df_fe = df.copy()
    for date_col in ['sanction_date', 'original_doc', 'revised_doc', 'snapshot_date']:
        if date_col in df_fe.columns:
            df_fe[f'{date_col}_dt'] = pd.to_datetime(df_fe[date_col], errors='coerce')
    
    if 'original_doc_dt' in df_fe.columns and 'sanction_date_dt' in df_fe.columns:
        df_fe['target_duration_months'] = np.maximum(1.0, (df_fe['original_doc_dt'] - df_fe['sanction_date_dt']).dt.days / 30.4375)
    else:
        df_fe['target_duration_months'] = 36.0
        
    if 'snapshot_date_dt' in df_fe.columns and 'sanction_date_dt' in df_fe.columns:
        df_fe['months_since_sanction'] = np.maximum(1.0, (df_fe['snapshot_date_dt'] - df_fe['sanction_date_dt']).dt.days / 30.4375)
    else:
        df_fe['months_since_sanction'] = 12.0
        
    df_fe['pct_time_elapsed'] = np.minimum(1.0, np.maximum(0.0, df_fe['months_since_sanction'] / df_fe['target_duration_months']))
    df_fe['expected_progress_pct'] = np.minimum(100.0, np.maximum(0.0, df_fe['pct_time_elapsed'] * 100.0))
    df_fe['physical_progress'] = pd.to_numeric(df_fe.get('physical_progress', 0), errors='coerce').fillna(0)
    df_fe['progress_lag_pct'] = df_fe['expected_progress_pct'] - df_fe['physical_progress']
    
    df_fe['original_cost'] = pd.to_numeric(df_fe.get('original_cost', 1), errors='coerce').fillna(1)
    df_fe['revised_cost'] = pd.to_numeric(df_fe.get('revised_cost', df_fe['original_cost']), errors='coerce').fillna(df_fe['original_cost'])
    df_fe['expenditure'] = pd.to_numeric(df_fe.get('expenditure', 0), errors='coerce').fillna(0)
    
    df_fe['cost_escalation_ratio'] = df_fe['revised_cost'] / np.maximum(1.0, df_fe['original_cost'])
    df_fe['expenditure_burn_rate'] = df_fe['expenditure'] / np.maximum(1.0, df_fe['months_since_sanction'])
    df_fe['financial_progress_pct'] = (df_fe['expenditure'] / np.maximum(1.0, df_fe['revised_cost'])) * 100.0
    df_fe['financial_physical_gap'] = df_fe['financial_progress_pct'] - df_fe['physical_progress']
    
    feature_cols = [
        'original_cost', 'revised_cost', 'expenditure', 'physical_progress',
        'target_duration_months', 'months_since_sanction', 'pct_time_elapsed',
        'expected_progress_pct', 'progress_lag_pct', 'cost_escalation_ratio',
        'expenditure_burn_rate', 'financial_progress_pct', 'financial_physical_gap'
    ]
    return df_fe, feature_cols

df_fe, feature_cols = engineer_real_features(df_raw)
print("Feature engineering complete. Prepared 13 real feature columns.")

In [ ]:
# Step 4: 80% Train | 10% Test Evaluation | 10% Held-Out User Test Set Split
from sklearn.model_selection import train_test_split

X = df_fe[feature_cols].fillna(0)
y_delay = df_fe.get('target_final_delay_months', df_fe['progress_lag_pct'] * 0.5).fillna(0)
y_cost = df_fe.get('target_cost_overrun_pct', (df_fe['cost_escalation_ratio'] - 1.0) * 100).fillna(0)
y_risk = df_fe.get('target_is_high_risk', (y_delay > 6) | (y_cost > 15)).astype(int)

X_train, X_temp, y_delay_train, y_delay_temp, y_cost_train, y_cost_temp, y_risk_train, y_risk_temp, df_train, df_temp = train_test_split(
    X, y_delay, y_cost, y_risk, df_raw, test_size=0.20, random_state=42
)
X_eval, X_holdout, y_delay_eval, y_delay_holdout, y_cost_eval, y_cost_holdout, y_risk_eval, y_risk_holdout, df_eval, df_holdout = train_test_split(
    X_temp, y_delay_temp, y_cost_temp, y_risk_temp, df_temp, test_size=0.50, random_state=42
)

print(f"Train Samples: {X_train.shape[0]} (80%)")
print(f"Evaluation Samples: {X_eval.shape[0]} (10%)")
print(f"Held-Out User Test Samples: {X_holdout.shape[0]} (10%)")

In [ ]:
# Step 5: Model Training & Metric Evaluation on Real Data
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.metrics import r2_score, mean_absolute_error, accuracy_score
import joblib

model_delay = RandomForestRegressor(n_estimators=100, max_depth=12, random_state=42, n_jobs=-1)
model_delay.fit(X_train, y_delay_train)
pred_delay = model_delay.predict(X_eval)

model_cost = RandomForestRegressor(n_estimators=100, max_depth=12, random_state=42, n_jobs=-1)
model_cost.fit(X_train, y_cost_train)
pred_cost = model_cost.predict(X_eval)

model_risk = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1)
model_risk.fit(X_train, y_risk_train)
pred_risk = model_risk.predict(X_eval)

print("=" * 60)
print("REAL MOSPI PAIMANA MODEL EVALUATION METRICS (10% Test Eval Set)")
print("=" * 60)
print(f"Schedule Delay Model R²: {r2_score(y_delay_eval, pred_delay):.3f} | MAE: {mean_absolute_error(y_delay_eval, pred_delay):.2f} months")
print(f"Cost Overrun Model R²:   {r2_score(y_cost_eval, pred_cost):.3f} | MAE: {mean_absolute_error(y_cost_eval, pred_cost):.2f}%")
print(f"Risk Tier Accuracy:     {accuracy_score(y_risk_eval, pred_risk) * 100:.1f}%")

In [ ]:
# Step 6: Export Real Model Artifacts & Held-Out Test Set for System Integration
import json
os.makedirs("models", exist_ok=True)
joblib.dump(model_delay, "models/paimana_delay_model.joblib")
joblib.dump(model_cost, "models/paimana_cost_model.joblib")
joblib.dump(model_risk, "models/paimana_risk_model.joblib")

df_holdout.to_csv("user_test_holdout.csv", index=False)
with open("user_test_holdout.json", "w") as f:
    json.dump(df_holdout.to_dict(orient="records"), f, indent=2)

print("[+] Exported user_test_holdout.csv and real models successfully.")
try:
    from google.colab import files
    files.download("user_test_holdout.csv")
    files.download("models/paimana_delay_model.joblib")
    files.download("models/paimana_cost_model.joblib")
    files.download("models/paimana_risk_model.joblib")
except Exception as e:
    print("Colab automatic download notice:", e)